# Carbon Mapper

The [Carbon Mapper](https://carbonmapper.org) reader provides typed
access to the Carbon Mapper STAC catalogue and plume API. The
upstream API ships methane and carbon-dioxide retrievals from
**Tanager-1**, **EMIT**, **AVIRIS-3**, **AVIRIS-NG**, and **GAO**.

This notebook walks through the typed query layer in
[`georeader.readers.carbonmapper`](../modules/readers_module.md#carbon-mapper-reader):

- **Typed models** — [`CMSource`](../modules/readers_module.md#carbon-mapper-reader)
  (DBSCAN-clustered point sources) and `CMTileItem` (a thin wrapper
  over a Carbon Mapper STAC item).
- **Raw HTTP wrappers** — `download.py` exposes
  `stac_get_item` / `stac_search` / `get_source_by_name` /
  `get_source_for_plume_name` etc., handling REST-vs-STAC bbox
  encoding, retries, and Bearer auth.
- **Typed query layer** — `api_queries.py` returns
  `CMRawPlume` / `CMTileItem` / `CMSource` instances, never raw
  dicts. Single-resource fetchers (`get_plume`, `get_tile`,
  `get_source`), list helpers (`list_*`), and cross-resolution
  helpers (`get_plume_context`, `list_tiles_for_source`).
- **Exception hierarchy** — `CMAPIError`, `CMPlumeNotFound`,
  `CMSourceNotFound`, `CMSceneNotPublished`. Single base class for
  catch-all, narrower types per resource.

The companion notebook
[`rasters_explore.ipynb`](rasters_explore.ipynb) covers the raster
wrappers (`CMImageRaster` / `CMPlumeRaster`) that consume the typed
items shown here.


## Install

The Carbon Mapper reader is gated behind the `[carbonmapper]` extra
to keep `georeader-spaceml`'s base install minimal. Install with:

```
pip install 'georeader-spaceml[carbonmapper]'
```

This pulls in `pydantic` (for `CMRawPlume`) and `requests` (for the
HTTP client). No Azure or other cloud-vendor SDKs are required.


## Authentication

Every cell below hits the live API and needs a Bearer token.
`CarbonMapperConfig.load()` resolves credentials in this priority
order:

1. **`CARBONMAPPER_TOKEN`** environment variable — one-shot, no
   refresh.
2. **`CARBONMAPPER_EMAIL` + `CARBONMAPPER_PASSWORD`** environment
   variables — refreshable via
   [`obtain_token`](https://api.carbonmapper.org/api/v1/docs).
3. **Config file** at one of:
   - `./config/carbonmapper_token.json`
   - `~/.config/carbonmapper/config.json`
   - `~/.carbonmapper.json`
   - `./.carbonmapper.json`

Sign up for a developer account at
[api.carbonmapper.org](https://api.carbonmapper.org) — the
free tier covers all the calls in this notebook.


## Setup

In [1]:
from datetime import datetime, timezone

from georeader.readers.carbonmapper import (
    CMAPIError,
    CMPlumeNotFound,
    CMSceneNotPublished,
    CMSource,
    CMSourceNotFound,
    CMTileItem,
    CarbonMapperConfig,
    api_queries,
    download,
)

# Resolve a Bearer token from env / config file (see "Authentication").
TOKEN = CarbonMapperConfig.load().refresh_access_token()

# Protagonist plume — Tanager-1 over the Permian basin, 2025-12-12.
PLUME_ID = "tan20251212t185057c20s4001-E"
SCENE_ID = PLUME_ID.rsplit("-", 1)[0]
PERMIAN_BBOX = (-104.5, 32.0, -103.5, 32.8)  # (W, S, E, N)
print(f"plume = {PLUME_ID}")
print(f"scene = {SCENE_ID}")

plume = tan20251212t185057c20s4001-E
scene = tan20251212t185057c20s4001


## 1 · Typed models

### 1.1 `CMSource` — DBSCAN-clustered point source

Carbon Mapper aggregates plumes detected at the same physical
location into a *source* — a deterministic point-source record
addressed by `{gas}_{sector}_{footprint_m}m_{lon}_{lat}`. The
`/catalog/sources.geojson` endpoint returns features whose
`source_name` carries a stray `?plume_gas=...` query suffix that
must be stripped before using the value as a key into other
endpoints. `CMSource.from_geojson_feature` does the strip
unconditionally — downstream code can treat `source_name` as
canonical.

In [2]:
feature = {
    "properties": {
        "source_name": "CH4_1B2_100m_-104.17525_32.49125?plume_gas=CH4&bbox=...",
        "sector": "1B2",
        "gas": "CH4",
        "plume_count": 12,
        "persistence": 0.42,
        "emission_auto": 250.0,
        "emission_uncertainty_auto": 35.0,
    },
    "geometry": {"type": "Point", "coordinates": [-104.17525, 32.49125]},
}
src = CMSource.from_geojson_feature(feature)
print(src.source_name)                        # suffix stripped
print((src.point.x, src.point.y))             # (-104.17525, 32.49125)
print(f"{src.plume_count} plumes · sector {src.sector} · gas {src.gas}")

CH4_1B2_100m_-104.17525_32.49125
(-104.17525, 32.49125)
12 plumes · sector 1B2 · gas CH4


### 1.2 `CMTileItem` — typed wrapper over a STAC item

Frozen dataclass exposing the fields we use in practice
(`scene_id`, `collection`, `datetime`, `platform`, `bbox`,
`geometry`, `asset_urls`). The full `properties` dict and the
`raw` STAC item stay attached for one-off field access.

In [3]:
stac_item = download.stac_get_item("l2b-ch4-mfa-v3a", SCENE_ID, token=TOKEN)
tile = CMTileItem.from_stac_item(stac_item)
print(f"{tile.scene_id} · {tile.platform} · {tile.datetime}")
print(f"bbox  : {tile.bbox}")
print(f"assets: {sorted(tile.asset_urls)[:5]}")

tan20251212t185057c20s4001 · tan · 2025-12-12 18:50:57+00:00
bbox  : (-104.5861937, 31.6662665, -103.9359726, 33.0604707)
assets: ['artifact-mask.tif', 'cmf-unortho.tif', 'cmf.tif', 'uas.txt', 'uncertainty-unortho.tif']


## 2 · `download.py` — raw HTTP wrappers

Thin endpoint wrappers — same return shape as the upstream JSON,
but with bbox-encoding, retries, and Bearer auth handled.

### 2.1 bbox encoding — REST vs STAC

Carbon Mapper's two API surfaces disagree on bbox shape:

- **REST Catalog** (`/catalog/...`) wants **repeated keys**:
  `?bbox=W&bbox=S&bbox=E&bbox=N`. Comma-joined returns 422.
- **STAC** (`/stac/...`) wants the **comma-joined** form:
  `?bbox=W,S,E,N`.

`_rest_bbox_params` returns a list-valued dict (`requests`
serialises lists as repeated keys); `_stac_bbox_param` returns the
comma-joined string.

In [4]:
from georeader.readers.carbonmapper.download import (
    _rest_bbox_params, _stac_bbox_param,
)

print(_rest_bbox_params(PERMIAN_BBOX))
# {'bbox': ['-104.5', '32.0', '-103.5', '32.8']}

print(_stac_bbox_param(PERMIAN_BBOX))
# {'bbox': '-104.5,32.0,-103.5,32.8'}

{'bbox': ['-104.5', '32.0', '-103.5', '32.8']}
{'bbox': '-104.5,32.0,-103.5,32.8'}


### 2.2 Endpoint wrappers

In [5]:
# stac_get_item — one STAC item by collection + scene_id
item = download.stac_get_item("l2b-ch4-mfa-v3a", SCENE_ID, token=TOKEN)
print(f"{item['id']} {item['properties']['datetime']}")

tan20251212t185057c20s4001 2025-12-12T18:50:57Z


In [6]:
# get_source_for_plume_name — find the source for our protagonist plume
src_dict = download.get_source_for_plume_name(PLUME_ID, token=TOKEN)
SOURCE_NAME = src_dict["source_name"]
print(SOURCE_NAME)

CH4_1B2_100m_-104.17525_32.49125


In [7]:
# get_source_by_name — REST source record (flat dict with properties)
src_dict = download.get_source_by_name(SOURCE_NAME, token=TOKEN)
print(f"plumes: {src_dict.get('plume_count')}  emission: {src_dict.get('emission_auto')} kg/h")

plumes: None  emission: None kg/h


In [8]:
# get_source_plumes_csv — every plume attributed to one source as CSV text
csv_text = download.get_source_plumes_csv(SOURCE_NAME, token=TOKEN)
print(csv_text[:200])

plume_id,plume_latitude,plume_longitude,datetime,country,state_province,ipcc_sector,gas,emission_cmf_type,plume_bounds,instrument,mission_phase,published_at,modified,emission_version,processing_softwa


In [9]:
# stac_search — accepts ids= for direct STAC item lookup
fc = download.stac_search(
    collections=["l2b-ch4-mfa-v3a"],
    ids=[SCENE_ID],
    limit=5,
    token=TOKEN,
)
print(f"{len(fc['features'])} feature(s) returned")

1 feature(s) returned


## 3 · `api_queries.py` — typed query layer

This is the layer downstream consumers (e.g. raster wrappers in
the [companion notebook](rasters_explore.ipynb)) build on. Three
families:

1. **Single-resource fetchers** — `get_plume`, `get_tile`,
   `get_source`. Translate 404s to typed exceptions.
2. **List helpers** — `list_plumes`, `list_tiles`,
   `list_sources`. Take a bbox + datetime range + filters.
3. **Cross-resolution** — given a plume, get its tile / source /
   full context. Given a source, get all its plumes / tiles.

### 3.1 Single-resource fetchers

In [10]:
plume = api_queries.get_plume(TOKEN, PLUME_ID)
print(f"{plume.plume_id}  gas={plume.gas}  emission_auto={plume.emission_auto} kg/h")
print(f"scene_id={plume.scene_id}")

tan20251212t185057c20s4001-E  gas=CH4  emission_auto=1007.6564374669618 kg/h
scene_id=64a51834-5fe5-40e0-aadd-e0c5944850c3


In [11]:
tile = api_queries.get_tile(TOKEN, SCENE_ID)  # default collection l2b-ch4-mfa-v3a
print(f"{tile.scene_id} · {tile.platform}")
print(f"bbox: {tile.bbox}")

tan20251212t185057c20s4001 · tan
bbox: (-104.5861937, 31.6662665, -103.9359726, 33.0604707)


In [12]:
source = api_queries.get_source(TOKEN, SOURCE_NAME)
print(f"{source.source_name}")
print(f"plumes: {source.plume_count}  sector: {source.sector}  emission: {source.emission_auto} kg/h")

CH4_1B2_100m_-104.17525_32.49125
plumes: 0  sector:   emission: None kg/h


### 3.2 Typed exceptions

404s are translated to typed `CMAPIError` subclasses — easy to
catch one resource type without swallowing real failures.

In [13]:
try:
    # Well-formed but non-existent plume id (the API 422s on malformed input).
    api_queries.get_plume(TOKEN, "tan29991231t000000c00s4001-Z")
except CMPlumeNotFound as exc:
    print(f"caught CMPlumeNotFound: {exc}")

try:
    api_queries.get_source(TOKEN, "CH4_1B2_100m_0_0")
except CMSourceNotFound as exc:
    print(f"caught CMSourceNotFound: {exc}")

try:
    # A scene whose L2B item is not yet published in STAC.
    api_queries.get_tile(TOKEN, "tan29991231t000000c00s4001")
except CMSceneNotPublished as exc:
    print(f"caught CMSceneNotPublished: {exc}")

# All three inherit from CMAPIError if you want a single catch.
print(f"isinstance(CMPlumeNotFound(...), CMAPIError) -> "
      f"{isinstance(CMPlumeNotFound('x'), CMAPIError)}")

caught CMPlumeNotFound: Plume not found: tan29991231t000000c00s4001-Z


caught CMSourceNotFound: Source not found: CH4_1B2_100m_0_0
caught CMSceneNotPublished: L2B scene not published: tan29991231t000000c00s4001
isinstance(CMPlumeNotFound(...), CMAPIError) -> True


### 3.3 List helpers

In [14]:
dt_min = datetime(2025, 12, 1, tzinfo=timezone.utc)
dt_max = datetime(2025, 12, 31, tzinfo=timezone.utc)

plumes = api_queries.list_plumes(
    TOKEN,
    bbox=PERMIAN_BBOX,
    datetime_min=dt_min,
    datetime_max=dt_max,
    gas="CH4",
)
print(f"{len(plumes)} plumes")
for p in plumes[:3]:
    print(f"  {p.plume_id}  emission_auto={p.emission_auto}")

22 plumes
  tan20251212t185057c20s4001-C  emission_auto=876.1038259991867
  tan20251212t185057c20s4001-D  emission_auto=432.5738771282001
  tan20251212t185057c20s4001-E  emission_auto=1007.6564374669618


In [15]:
# NB: STAC search caps page size at 100 — pass an explicit limit until
# pagination lands.
tiles = api_queries.list_tiles(
    TOKEN,
    bbox=PERMIAN_BBOX,
    datetime_min=dt_min,
    datetime_max=dt_max,
    limit=50,
)
print(f"{len(tiles)} tiles")
for t in tiles[:3]:
    print(f"  {t.scene_id} {t.platform} {t.datetime}")

4 tiles
  tan20251212t185057c20s4001 tan 2025-12-12 18:50:57+00:00
  tan20251210t183649c71s4001 tan 2025-12-10 18:36:49+00:00
  tan20251210t183749c38s4001 tan 2025-12-10 18:37:49+00:00


In [16]:
sources = api_queries.list_sources(TOKEN, bbox=PERMIAN_BBOX, gas="CH4")
print(f"{len(sources)} sources")
for s in sources[:3]:
    print(f"  {s.source_name}  plumes={s.plume_count}  emission={s.emission_auto}")

10513 sources
  CH4_6A_500m_-117.26768_34.59375  plumes=3  emission=23.0636206375622
  CH4_6A_500m_-118.51708_34.32770  plumes=484  emission=1082.9529787100435
  CH4_6A_500m_-119.38080_36.39176  plumes=34  emission=174.91747754329324


### 3.4 Cross-resolution helpers

The headline value-add — these do the join work the upstream API
leaves to the caller (scene-id derivation, source lookup-by-plume,
dedup of `scene_id`s per source, etc.).

In [17]:
# plume → tile (returns None when the L2B scene isn't published yet)
tile = api_queries.get_tile_for_plume(TOKEN, PLUME_ID)
print(f"tile  : {tile and tile.scene_id}")

tile  : tan20251212t185057c20s4001


In [18]:
# plume → source (returns None when CM hasn't clustered this plume yet)
source = api_queries.get_source_for_plume(TOKEN, PLUME_ID)
print(f"source: {source and source.source_name}")

source: CH4_1B2_100m_-104.17525_32.49125


In [19]:
# One call → (plume, tile|None, source|None) — the typical ingestion shape.
plume, tile, source = api_queries.get_plume_context(TOKEN, PLUME_ID)
print(f"plume  : {plume.plume_id}")
print(f"  tile  : {tile and tile.scene_id}")
print(f"  source: {source and source.source_name}")

plume  : tan20251212t185057c20s4001-E
  tile  : tan20251212t185057c20s4001
  source: CH4_1B2_100m_-104.17525_32.49125


In [20]:
# tile → all plumes in that scene
plumes = api_queries.list_plumes_for_tile(TOKEN, SCENE_ID)
print(f"{len(plumes)} plumes in scene {SCENE_ID}")

0 plumes in scene tan20251212t185057c20s4001


In [21]:
# source → every plume attributed to it (parsed from CSV)
plumes = api_queries.list_plumes_for_source(TOKEN, SOURCE_NAME)
print(f"{len(plumes)} plumes for source {SOURCE_NAME}")

# source → distinct parent tiles (dedups scene_ids before STAC ids= search)
tiles = api_queries.list_tiles_for_source(TOKEN, SOURCE_NAME)
print(f"{len(tiles)} unique parent tiles")

1 plumes for source CH4_1B2_100m_-104.17525_32.49125


1 unique parent tiles


## 4 · End-to-end mini-workflow

Tie it together: take a bbox + date range, list plumes, expand
each into full context, count how many have a published L2B parent
and how many are clustered into a source.

In [22]:
plumes = api_queries.list_plumes(
    TOKEN, bbox=PERMIAN_BBOX, datetime_min=dt_min, datetime_max=dt_max, gas="CH4",
)

n_with_tile = n_with_source = 0
for p in plumes[:25]:  # cap to keep the request count modest
    _, tile, source = api_queries.get_plume_context(TOKEN, p.plume_id)
    n_with_tile += tile is not None
    n_with_source += source is not None

print(f"checked       : {min(len(plumes), 25)}")
print(f"L2B published : {n_with_tile}")
print(f"in a source   : {n_with_source}")

checked       : 22
L2B published : 22
in a source   : 22


## See also

- [`rasters_explore.ipynb`](rasters_explore.ipynb) — `CMImageRaster` /
  `CMPlumeRaster`, georeader-backed lazy raster wrappers that
  consume the typed items returned here.
- [Carbon Mapper Reader API reference](../modules/readers_module.md#carbon-mapper-reader)
  — full module / class / function listing rendered from
  source.
- [Carbon Mapper API docs](https://api.carbonmapper.org/api/v1/docs)
  — upstream OpenAPI schema and endpoint inventory.
